**1. Import Libraries**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import random

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


**ADD MOBILENETV1+BIGRU MODEL CODE**

In [ ]:
# just run the data preprocessing & the model definition, dont run the model.fit part, that is just for reference

**2. Data PreProcessing**

In [ ]:
# Load
df = pd.read_csv('/content/drive/MyDrive/5G_NIDD_multiclass_clean.csv', low_memory=False)

print("Original shape:", df.shape)

# Target
y = df['Label']
X = df.drop(columns=['Label', 'Attack Type', 'Attack Tool'], errors='ignore')

# Remove obvious non-learning columns
drop_cols = [
    'SrcMac','DstMac','SrcAddr','DstAddr','StartTime','LastTime',        # drop these columns as they dont have any importance in model(these are IP addresses)
    'SrcOui','DstOui'
]

X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors='ignore')

# Keep only numeric features
X = X.select_dtypes(include=[np.number])   # keeps only numeric features

print("After numeric selection:", X.shape)


Original shape: (1215890, 112)
After numeric selection: (1215890, 86)


In [ ]:
X.replace([np.inf, -np.inf], np.nan, inplace=True)     # remove infinite values
X.fillna(0, inplace=True)                              # handle missing values


In [ ]:
selector = SelectKBest(score_func=f_classif, k=36)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected Features:", selected_features.tolist())


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 1  7 13 14 19 20 21 22 23 24 25 26 27 34 35 36 37 48 49 50 51 57 58 59
 60 61 62 63 64 65 66 67 68 69 70 71 72 77 78 79 80] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Selected Features: ['Rank', 'Seq', 'Dur', 'RunTime', 'Mean', 'Sum', 'Min', 'Max', 'sTos', 'dTos', 'sTtl', 'dTtl', 'sHops', 'dHops', 'TotPkts', 'SrcPkts', 'DstPkts', 'TotBytes', 'SrcBytes', 'DstBytes', 'Offset', 'sMeanPktSz', 'dMeanPktSz', 'Loss', 'SrcLoss', 'DstLoss', 'pLoss', 'SrcWin', 'DstWin', 'sVid', 'dVid', 'SrcTCPBase', 'DstTCPBase', 'TcpRtt', 'SynAck', 'AckDat']


In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

num_classes = len(np.unique(y_encoded))
print("Classes:", num_classes)


Classes: 20


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_encoded,
    test_size=0.2,                    # train data = 80%, test data = 20%
    stratify=y_encoded,
    random_state=42
)


In [ ]:
from sklearn.preprocessing import QuantileTransformer

scaler = QuantileTransformer(
    n_quantiles=min(1000, len(X_train)),   # avoids warning if dataset < 1000
    output_distribution='normal',          # 'normal' usually works better for neural networks
    random_state=42
)

X_train = scaler.fit_transform(X_train)   # FIT ONLY TRAIN
X_test  = scaler.transform(X_test)        # TRANSFORM TEST

In [ ]:
X_train = X_train.reshape(-1, 36, 1)        # Prepare data in 3D format required by GRU networks
X_test  = X_test.reshape(-1, 36, 1)


**3. MobileNetV1 + BIGRU Model**

In [ ]:
def MobileNetV1_BiGRU(drop_rate=0.30000000000000004, gru_units=128, dense_units=384):

    inp = Input(shape=(36,1))

    # ----- CNN BRANCH -----                    # MOBILE-NET V1 MODEL
    x = Reshape((36,1,1))(inp)

    x = Conv2D(32,(3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(64,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(128,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    cnn_out = GlobalAveragePooling2D()(x)

    # ----- GRU BRANCH -----                                 # BIGRU MODEL
    r = Bidirectional(GRU(gru_units, return_sequences=True))(inp)
    r = Bidirectional(GRU(gru_units))(r)

    # ----- FUSION -----                                    # PROJECTION(Combines both models)
    merged = Concatenate()([cnn_out, r])

    merged = Dense(dense_units, activation="relu")(merged)
    merged = Dense(dense_units//2, activation="relu")(merged)
    merged = Dropout(drop_rate)(merged)

    out = Dense(num_classes, activation="softmax")(merged)

    model = Model(inp, out)
    return model


In [ ]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.7 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf

def focal_loss(gamma=1.0, alpha=0.30000000000000004):

    def loss(y_true, y_pred):

        y_true = tf.cast(y_true, tf.int32)
        y_true = tf.one_hot(y_true, depth=num_classes)

        ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)

        pt = tf.exp(-ce)
        focal = alpha * tf.pow(1 - pt, gamma) * ce

        return tf.reduce_mean(focal)

    return loss


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights))
print(class_weights)


{np.int64(0): np.float64(0.64811172410117), np.int64(1): np.float64(0.6491671115856914), np.int64(2): np.float64(3.325965944060726), np.int64(3): np.float64(4.20650406504065), np.int64(4): np.float64(37.819284603421465), np.int64(5): np.float64(44.74296228150874), np.int64(6): np.float64(1.3619983757596124), np.int64(7): np.float64(4.309374446216552), np.int64(8): np.float64(5.309563318777292), np.int64(9): np.float64(5.274438781043271), np.int64(10): np.float64(1.9601644365629534), np.int64(11): np.float64(4.803516049382716), np.int64(12): np.float64(5.220652640618291), np.int64(13): np.float64(5.217292426517915), np.int64(14): np.float64(1.5948189926547744), np.int64(15): np.float64(1.9197000197355436), np.int64(16): np.float64(0.12998123867505493), np.int64(17): np.float64(0.21242149215139894), np.int64(18): np.float64(6.053721682847897), np.int64(19): np.float64(5.8995147986414365)}


In [ ]:
def get_compiled_model():

    model = MobileNetV1_BiGRU()

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.0022052893576960772),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

**4. Create Federated Clients**

In [ ]:
NUM_CLIENTS = 3

client_data = {}

X_split = np.array_split(X_train, NUM_CLIENTS)          # (training) Data is distributed almost equally between the 5 clients
y_split = np.array_split(y_train, NUM_CLIENTS)

for i in range(NUM_CLIENTS):
    client_data[i] = (X_split[i], y_split[i])

**5. Client Local Training Function**

In [ ]:
def client_update(global_weights, dataset, local_epochs=2):

    # compiled model
    local_model = get_compiled_model()

    # load global weights
    local_model.set_weights(global_weights)

    X, y = dataset

    local_model.fit(
        X,
        y,
        epochs=local_epochs,
        batch_size=32,
        verbose=0
    )

    return local_model.get_weights()

**6. Federated Averaging (FedAvg)**

In [ ]:
def aggregate_weights(client_weights):

    avg_weights = []

    for i, weights_list_tuple in enumerate(zip(*client_weights)):
        layer_avg = np.mean(weights_list_tuple, axis=0)

        print(f"\nLayer {i} aggregation:")
        print(f"Shape: {layer_avg.shape}")
        print(f"Mean: {np.mean(layer_avg):.6f}")
        print(f"Std: {np.std(layer_avg):.6f}")

        avg_weights.append(layer_avg)

    return avg_weights

**7. Federated Training Loop (Server)**

In [ ]:
ROUNDS = 10
CLIENTS_PER_ROUND = 3

# global model (compiled)
global_model = get_compiled_model()

# load pretrained weights
global_model.load_weights("optuna_best_weights.h5")

weights = global_model.get_weights()

print("First layer mean:", np.mean(weights[0]))

for round_num in range(ROUNDS):

    print("\nFL Round:", round_num)

    global_weights = global_model.get_weights()

    client_weights = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:

        weights = client_update(
            global_weights,
            client_data[client]
        )

        client_weights.append(weights)

    new_global_weights = federated_average(client_weights)

    global_model.set_weights(new_global_weights)

    # ADD THIS
    loss, acc = global_model.evaluate(X_test, y_test, verbose=1)

    print(f"Global Accuracy after round {round_num}: {acc:.4f}")


FL Round: 0
7600/7600 ━━━━━━━━━━━━━━━━━━━━ 53s 7ms/step - accuracy: 0.5889 - loss: 1.1128
Global Accuracy after round 0: 0.5889

FL Round: 1
7600/7600 ━━━━━━━━━━━━━━━━━━━━ 52s 7ms/step - accuracy: 0.8922 - loss: 0.2540
Global Accuracy after round 1: 0.8922

FL Round: 2
7600/7600 ━━━━━━━━━━━━━━━━━━━━ 52s 7ms/step - accuracy: 0.8977 - loss: 0.2160
Global Accuracy after round 2: 0.8977

FL Round: 3
7600/7600 ━━━━━━━━━━━━━━━━━━━━ 51s 7ms/step - accuracy: 0.9178 - loss: 0.1860
Global Accuracy after round 3: 0.9178

FL Round: 4
7600/7600 ━━━━━━━━━━━━━━━━━━━━ 52s 7ms/step - accuracy: 0.9024 - loss: 0.1972
Global Accuracy after round 4: 0.9024

FL Round: 5
7600/7600 ━━━━━━━━━━━━━━━━━━━━ 53s 7ms/step - accuracy: 0.9233 - loss: 0.1742
Global Accuracy after round 5: 0.9233

FL Round: 6
7600/7600 ━━━━━━━━━━━━━━━━━━━━ 52s 7ms/step - accuracy: 0.9289 - loss: 0.1680
Global Accuracy after round 6: 0.9289

FL Round: 7


KeyboardInterrupt: 

**8. Evaluate Global Model**

In [ ]:
loss, accuracy = global_model.evaluate(X_test, y_test)
print("Final Global Model Accuracy:", accuracy)

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 61s 8ms/step - accuracy: 0.9289 - loss: 0.1680
Final Global Model Accuracy: 0.9289491772651672
